# Create PySpark Engine

In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("Fraud Detection") \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/11/03 15:43:30 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


# Load CSV Dataset

In [2]:
df = spark.read.csv("/Users/hamoaster/Downloads/Datawarehouse/onlinefraud.csv", header=True, inferSchema=True)

# Show Dataset & Understanding It

In [3]:
df.show(5)

+----+--------+--------+-----------+-------------+--------------+-----------+--------------+--------------+-------+--------------+
|step|    type|  amount|   nameOrig|oldbalanceOrg|newbalanceOrig|   nameDest|oldbalanceDest|newbalanceDest|isFraud|isFlaggedFraud|
+----+--------+--------+-----------+-------------+--------------+-----------+--------------+--------------+-------+--------------+
|   1| PAYMENT| 9839.64|C1231006815|     170136.0|     160296.36|M1979787155|           0.0|           0.0|      0|             0|
|   1| PAYMENT| 1864.28|C1666544295|      21249.0|      19384.72|M2044282225|           0.0|           0.0|      0|             0|
|   1|TRANSFER|   181.0|C1305486145|        181.0|           0.0| C553264065|           0.0|           0.0|      1|             0|
|   1|CASH_OUT|   181.0| C840083671|        181.0|           0.0|  C38997010|       21182.0|           0.0|      1|             0|
|   1| PAYMENT|11668.14|C2048537720|      41554.0|      29885.86|M1230701703|      

In [4]:
df.printSchema()

root
 |-- step: integer (nullable = true)
 |-- type: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- nameOrig: string (nullable = true)
 |-- oldbalanceOrg: double (nullable = true)
 |-- newbalanceOrig: double (nullable = true)
 |-- nameDest: string (nullable = true)
 |-- oldbalanceDest: double (nullable = true)
 |-- newbalanceDest: double (nullable = true)
 |-- isFraud: integer (nullable = true)
 |-- isFlaggedFraud: integer (nullable = true)



# Pre-Porcessing

### Dropping Dupes

In [5]:
df = df.dropDuplicates()
df.show(5)

24/11/03 15:43:42 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
24/11/03 15:43:42 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
24/11/03 15:43:42 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
24/11/03 15:43:42 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
24/11/03 15:43:42 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
24/11/03 15:43:42 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
24/11/03 15:43:42 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
24/11/03 15:43:42 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
24/11/03 15:43:42 WARN RowBasedKeyValueBatch: Calling spill() on

+----+--------+---------+-----------+-------------+--------------+-----------+--------------+--------------+-------+--------------+
|step|    type|   amount|   nameOrig|oldbalanceOrg|newbalanceOrig|   nameDest|oldbalanceDest|newbalanceDest|isFraud|isFlaggedFraud|
+----+--------+---------+-----------+-------------+--------------+-----------+--------------+--------------+-------+--------------+
|   1|CASH_OUT|210370.09|C2121995675|          0.0|           0.0|C1170794006|    1442298.03|      22190.99|      0|             0|
|   1|CASH_OUT| 89962.11|C2018260103|          0.0|           0.0| C240650537|     157012.19|           0.0|      0|             0|
|   1| CASH_IN|418688.27|C1756819670|   1888091.55|    2306779.82| C401424608|     956455.03|    1178808.14|      0|             0|
|   1| CASH_IN| 127954.7|C1930837320|     373873.7|      501828.4| C564160838|    1590611.34|    1254956.07|      0|             0|
|   1| PAYMENT|   7797.0| C500230084|       816.91|           0.0|M202302684

### Nulls

##### Check Nulls

In [6]:
from pyspark.sql import functions as F
missing_values = df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns])
missing_values.show()

24/11/03 15:43:53 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
24/11/03 15:43:53 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
24/11/03 15:43:53 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
24/11/03 15:43:53 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
24/11/03 15:43:53 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
24/11/03 15:43:53 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
24/11/03 15:43:53 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
24/11/03 15:43:53 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
24/11/03 15:43:53 WARN RowBasedKeyValueBatch: Calling spill() on

+----+----+------+--------+-------------+--------------+--------+--------------+--------------+-------+--------------+
|step|type|amount|nameOrig|oldbalanceOrg|newbalanceOrig|nameDest|oldbalanceDest|newbalanceDest|isFraud|isFlaggedFraud|
+----+----+------+--------+-------------+--------------+--------+--------------+--------------+-------+--------------+
|   0|   0|     0|       0|            0|             0|       0|             0|             0|      0|             0|
+----+----+------+--------+-------------+--------------+--------+--------------+--------------+-------+--------------+



No nulls were detected!

### Data Conversion

IsFraud & IsFlaggedFraud are integers as we saw earlier. However I was considering to convert them to boolean in new columns just to have more options in our hands 

In [7]:
df = df.withColumn("isFraud_Boolean", (df["isFraud"] == 1).cast("boolean"))
df = df.withColumn("isFlaggedFraud_Boolean", (df["isFlaggedFraud"] == 1).cast("boolean"))
df.show(5)

24/11/03 15:44:06 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
24/11/03 15:44:06 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
24/11/03 15:44:06 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
24/11/03 15:44:06 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
24/11/03 15:44:06 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
24/11/03 15:44:06 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
24/11/03 15:44:06 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
24/11/03 15:44:06 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
24/11/03 15:44:06 WARN RowBasedKeyValueBatch: Calling spill() on

+----+--------+---------+-----------+-------------+--------------+-----------+--------------+--------------+-------+--------------+---------------+----------------------+
|step|    type|   amount|   nameOrig|oldbalanceOrg|newbalanceOrig|   nameDest|oldbalanceDest|newbalanceDest|isFraud|isFlaggedFraud|isFraud_Boolean|isFlaggedFraud_Boolean|
+----+--------+---------+-----------+-------------+--------------+-----------+--------------+--------------+-------+--------------+---------------+----------------------+
|   1|CASH_OUT|210370.09|C2121995675|          0.0|           0.0|C1170794006|    1442298.03|      22190.99|      0|             0|          false|                 false|
|   1|CASH_OUT| 89962.11|C2018260103|          0.0|           0.0| C240650537|     157012.19|           0.0|      0|             0|          false|                 false|
|   1| CASH_IN|418688.27|C1756819670|   1888091.55|    2306779.82| C401424608|     956455.03|    1178808.14|      0|             0|          fals

### Create ID

This dataset clearly does not have an ID. So we will create that right away!

In [10]:
from pyspark.sql.functions import monotonically_increasing_id
df = df.withColumn("Transaction_ID", monotonically_increasing_id())
df.show(5)

24/11/03 15:46:30 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
24/11/03 15:46:30 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
24/11/03 15:46:30 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
24/11/03 15:46:30 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
24/11/03 15:46:30 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
24/11/03 15:46:30 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
24/11/03 15:46:30 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
24/11/03 15:46:31 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
24/11/03 15:46:31 WARN RowBasedKeyValueBatch: Calling spill() on

+----+--------+---------+-----------+-------------+--------------+-----------+--------------+--------------+-------+--------------+---------------+----------------------+--------------+
|step|    type|   amount|   nameOrig|oldbalanceOrg|newbalanceOrig|   nameDest|oldbalanceDest|newbalanceDest|isFraud|isFlaggedFraud|isFraud_Boolean|isFlaggedFraud_Boolean|Transaction_ID|
+----+--------+---------+-----------+-------------+--------------+-----------+--------------+--------------+-------+--------------+---------------+----------------------+--------------+
|   1|CASH_OUT|210370.09|C2121995675|          0.0|           0.0|C1170794006|    1442298.03|      22190.99|      0|             0|          false|                 false|             0|
|   1|CASH_OUT| 89962.11|C2018260103|          0.0|           0.0| C240650537|     157012.19|           0.0|      0|             0|          false|                 false|             1|
|   1| CASH_IN|418688.27|C1756819670|   1888091.55|    2306779.82| C40

# Transformation Using Star Schema

### Fact Table

Fact Table: contains info about transactions

Columns: Transaction_ID, step, type, amount, oldbalanceOrg, newbalanceOrig, oldbalanceDest, newbalanceDest, isFraud, isFlaggedFraud

In [11]:
fact_table = df.select(
    F.col("Transaction_ID"),
    F.col("step"),
    F.col("type"),
    F.col("amount"),
    F.col("oldbalanceOrgin"),
    F.col("newbalanceOrigin"),
    F.col("oldbalanceDestination"),
    F.col("newbalanceDestination"),
    F.col("isFraud"),
    F.col("isFlaggedFraud")
)

### Dimensions Table

##### Customers

Customers will be split it into customers_origin and customers_destination. The Origin will represent the payer's transactions. The destination will represent the receiver's transaction

Columns: customer_id (derived from nameOrig and nameDest), oldbalance, newbalance

In [26]:
customers_origin = df.select(
    F.col("nameOrig").alias("nameOrig"),  
    F.col("oldbalanceOrg").alias("oldbalance"),  
    F.col("newbalanceOrig").alias("newbalance")  
).distinct()

In [27]:
customers_destination = df.select(
    F.col("nameDest").alias("customer_id"),  
    F.col("oldbalanceDest").alias("oldbalance"),  
    F.col("newbalanceDest").alias("newbalance")  
).distinct()

##### Transaction Types

Contains the types of transactions
Columns: transaction_type_id, type

In [21]:
transaction_types = df.select(F.col("type").alias("transaction_type")).distinct() \
                       .withColumn("transaction_type_id", F.monotonically_increasing_id())

### Display The Tables

In [28]:
print('Fact Table:')
fact_table.show(5)
print('Customer Origin Table:')
customers_origin.show(5)
print('Customer Destination Table:')
customers_destination.show(5)
print('Transaction Table:')
transaction_types.show(5)

Fact Table:


24/11/03 16:18:24 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
24/11/03 16:18:24 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
24/11/03 16:18:24 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
24/11/03 16:18:24 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
24/11/03 16:18:24 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
24/11/03 16:18:24 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
24/11/03 16:18:24 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
24/11/03 16:18:24 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
24/11/03 16:18:24 WARN RowBasedKeyValueBatch: Calling spill() on

+--------------+----+--------+---------+-------------+--------------+--------------+--------------+-------+--------------+
|Transaction_ID|step|    type|   amount|oldbalanceOrg|newbalanceOrig|oldbalanceDest|newbalanceDest|isFraud|isFlaggedFraud|
+--------------+----+--------+---------+-------------+--------------+--------------+--------------+-------+--------------+
|             0|   1|CASH_OUT|210370.09|          0.0|           0.0|    1442298.03|      22190.99|      0|             0|
|             1|   1|CASH_OUT| 89962.11|          0.0|           0.0|     157012.19|           0.0|      0|             0|
|             2|   1| CASH_IN|418688.27|   1888091.55|    2306779.82|     956455.03|    1178808.14|      0|             0|
|             3|   1| CASH_IN| 127954.7|     373873.7|      501828.4|    1590611.34|    1254956.07|      0|             0|
|             4|   1| PAYMENT|   7797.0|       816.91|           0.0|           0.0|           0.0|      0|             0|
+--------------+

24/11/03 16:18:34 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
24/11/03 16:18:34 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
24/11/03 16:18:34 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
24/11/03 16:18:34 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
24/11/03 16:18:34 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
24/11/03 16:18:34 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.


+-----------+----------+----------+
|   nameOrig|oldbalance|newbalance|
+-----------+----------+----------+
| C867093003|       0.0|       0.0|
|C1293462056|7679741.48|7964926.82|
| C745627268|   16741.0|  15057.05|
| C288919635|1330599.59|1329839.38|
| C437630857|   5881.44|       0.0|
+-----------+----------+----------+
only showing top 5 rows

Customer Destination Table:


24/11/03 16:18:39 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
24/11/03 16:18:39 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
24/11/03 16:18:39 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
24/11/03 16:18:39 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
24/11/03 16:18:39 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
24/11/03 16:18:39 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.


+-----------+----------+----------+
|customer_id|oldbalance|newbalance|
+-----------+----------+----------+
|C2083562754| 278846.75|1186556.81|
|M1272051933|       0.0|       0.0|
|M1650113431|       0.0|       0.0|
| C459857341|   42211.0| 387263.02|
| C932583850| 696184.96|2719172.89|
+-----------+----------+----------+
only showing top 5 rows

Transaction Table:


+----------------+-------------------+
|transaction_type|transaction_type_id|
+----------------+-------------------+
|        TRANSFER|                  0|
|         CASH_IN|                  1|
|        CASH_OUT|                  2|
|         PAYMENT|                  3|
|           DEBIT|                  4|
+----------------+-------------------+



# Load To MySQL

In [30]:
jdbc_url = "jdbc:mysql://localhost:3306/Data_Warehousing"
properties = {
    "user": "root",
    "driver": "com.mysql.cj.jdbc.Driver"  
}

In [31]:
fact_table.write.jdbc(url=jdbc_url, table="fact_table", mode="overwrite", properties=properties)
customers_origin.write.jdbc(url=jdbc_url, table="customers_origin", mode="overwrite", properties=properties)
customers_destination.write.jdbc(url=jdbc_url, table="customers_destination", mode="overwrite", properties=properties)
transaction_types.write.jdbc(url=jdbc_url, table="transaction_types", mode="overwrite", properties=properties)

Py4JJavaError: An error occurred while calling o255.jdbc.
: java.lang.ClassNotFoundException: com.mysql.cj.jdbc.Driver
	at java.base/java.net.URLClassLoader.findClass(URLClassLoader.java:476)
	at java.base/java.lang.ClassLoader.loadClass(ClassLoader.java:594)
	at java.base/java.lang.ClassLoader.loadClass(ClassLoader.java:527)
	at org.apache.spark.sql.execution.datasources.jdbc.DriverRegistry$.register(DriverRegistry.scala:46)
	at org.apache.spark.sql.execution.datasources.jdbc.JDBCOptions.$anonfun$driverClass$1(JDBCOptions.scala:103)
	at org.apache.spark.sql.execution.datasources.jdbc.JDBCOptions.$anonfun$driverClass$1$adapted(JDBCOptions.scala:103)
	at scala.Option.foreach(Option.scala:407)
	at org.apache.spark.sql.execution.datasources.jdbc.JDBCOptions.<init>(JDBCOptions.scala:103)
	at org.apache.spark.sql.execution.datasources.jdbc.JdbcOptionsInWrite.<init>(JDBCOptions.scala:254)
	at org.apache.spark.sql.execution.datasources.jdbc.JdbcOptionsInWrite.<init>(JDBCOptions.scala:258)
	at